# Precision experiment — analysis with pyxations

Parses WebGazer CSVs, converts to BIDS, runs REMoDNaV,
and propagates trial metadata via `behavioral_columns`.


## 0. Setup

In [ ]:
# IMPORTANT: use python3.10
# In terminal: python3.10 -m jupyter notebook

# Install pyxations from local source (uncomment on first run):
!pip install -e /home/gus/Documents/REPOS/pyxations

from pathlib import Path
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow.feather as feather

import pyxations as pyx

print('pyxations', pyx.__version__ if hasattr(pyx, '__version__') else 'OK')

REPO_ROOT = Path('.').resolve().parent  # notebook lives inside the experiment subfolder
print(f'Working dir: {REPO_ROOT}')


### 1 Prepare files for pyxations

Files already follow the format `{subject}_{session}.csv`  
(e.g. `gus_webcam4.csv`, `juan_webcam9_repe.csv`).


In [ ]:
RAW_PREC = REPO_ROOT / "precision_experiment" / "raw_data" / "precision-antisaccade-exp"
PREC_STAGING = REPO_ROOT / "precision_experiment" / "_staging"
PREC_STAGING.mkdir(exist_ok=True)

raw_files = sorted(RAW_PREC.glob("*.csv"))
print(f"Files found: {len(raw_files)}")
print([f.name for f in raw_files])


In [ ]:
# Copy to staging (names already follow subject_session.csv format)
for csv_file in raw_files:
    dest = PREC_STAGING / csv_file.name
    if not dest.exists():
        shutil.copy(csv_file, dest)

print(f"Files in staging: {len(list(PREC_STAGING.glob('*.csv')))}")


### 2 Convert to BIDS and compute derivatives


In [ ]:
PREC_BIDS = REPO_ROOT / 'precision_experiment' / 'bids'

prec_bids_path = pyx.dataset_to_bids(
    target_folder_path=PREC_BIDS,
    files_folder_path=PREC_STAGING,
    dataset_name='precision',
    session_substrings=2,
    format_name='webgazer',
)

prec_derivatives_path = pyx.compute_derivatives_for_dataset(
    bids_dataset_folder=prec_bids_path,
    dataset_format='webgazer',
    detection_algorithm='remodnav',
    num_processes=4,
    behavioral_columns=['trial-tag', 'center_x', 'center_y', 'start-x', 'start-y'],
    overwrite=True,
)
print(f'Derivatives at: {prec_derivatives_path}')


In [ ]:
# Mapping subject_id (BIDS) -> old_id (original name)
participants = pd.read_csv(prec_bids_path / 'participants.tsv', sep='\t', dtype=str)
prec_id_map = dict(zip(participants['subject_id'], participants['old_subject_id']))
print('Subjects:', prec_id_map)


### 3 Precision analysis

Columns `trial-tag`, `center_x`, `center_y`, `start-x`, `start-y`  
are already in `samples.feather` thanks to `behavioral_columns`.  
The function reads no CSV — it works entirely with the pyxations feather.


In [ ]:
def analyze_precision_subject(samples_path,
                               trial_tag='validation-stimulus',
                               first_sample=500, max_var=75):
    """
    Precision metrics using pyxations samples.
    Columns trial-tag, center_x, center_y, start-x, start-y
    are already in samples.feather — no CSV is read.
    """
    samples = feather.read_feather(samples_path)

    # Validation trials directly from samples
    val_samples = samples[samples['trial-tag'] == trial_tag]
    val_trials = (
        val_samples[['trial_index', 'center_x', 'center_y', 'start-x', 'start-y']]
        .drop_duplicates('trial_index')
    )

    results = []
    for _, row in val_trials.iterrows():
        tidx        = row['trial_index']
        presented_x = row['center_x'] + row['start-x']
        presented_y = row['center_y'] + row['start-y']

        trial_samps = samples[
            (samples['trial_index'] == tidx) & (samples['tSample'] > first_sample)
        ]
        if trial_samps.empty:
            results.append(dict(trial_index=tidx, presented_point_x=presented_x,
                                presented_point_y=presented_y, n_samples=0,
                                horizontal_error_px=np.nan, vertical_error_px=np.nan,
                                total_error_px=np.nan))
            continue

        xs = trial_samps['X'].values
        ys = trial_samps['Y'].values

        if np.std(xs) > max_var or np.std(ys) > max_var:
            results.append(dict(trial_index=tidx, presented_point_x=presented_x,
                                presented_point_y=presented_y, n_samples=len(trial_samps),
                                horizontal_error_px=np.nan, vertical_error_px=np.nan,
                                total_error_px=np.nan))
            continue

        results.append(dict(
            trial_index=tidx,
            presented_point_x=presented_x,
            presented_point_y=presented_y,
            n_samples=len(trial_samps),
            horizontal_error_px=np.abs(xs - presented_x).mean(),
            vertical_error_px=np.abs(ys - presented_y).mean(),
            total_error_px=np.sqrt((xs - presented_x)**2 + (ys - presented_y)**2).mean(),
        ))

    return pd.DataFrame(results)


In [ ]:
all_prec_results = {}

for subject_id, old_id in prec_id_map.items():
    sub_path = prec_derivatives_path / f'sub-{subject_id}'
    if not sub_path.exists():
        print(f'sub-{subject_id} not found, skipping.')
        continue

    for ses_dir in sorted(sub_path.iterdir()):
        if not ses_dir.name.startswith('ses-'):
            continue
        ses_name = ses_dir.name[4:]
        samples_path = ses_dir / 'samples.feather'
        if not samples_path.exists():
            print(f'  No samples for sub-{subject_id} ses-{ses_name}, skipping.')
            continue

        print(f'Processing sub-{subject_id} ({old_id}), ses-{ses_name}...')
        # no pd.read_csv — all metadata comes from the feather
        df_res = analyze_precision_subject(samples_path)
        df_res['subject_id'] = subject_id
        df_res['old_id'] = old_id
        df_res['session'] = ses_name

        key = f'{old_id}_{ses_name}'
        all_prec_results[key] = df_res

        n_valid = df_res['horizontal_error_px'].notna().sum()
        print(f'  Valid trials: {n_valid}/{len(df_res)}')
        print(f'  Horizontal error: {df_res["horizontal_error_px"].mean():.1f} px')
        print(f'  Vertical error:   {df_res["vertical_error_px"].mean():.1f} px')
        print(f'  Total error:      {df_res["total_error_px"].mean():.1f} px\n')


### 4 Error heatmap per validation point

Replicates the heatmap from the original `precision_analysis.ipynb`.  
Each cell shows the mean error for that point in the 3×3 grid.


In [ ]:
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms


def confidence_ellipse(x, y, ax, n_std=3.0, facecolor="none", **kwargs):
    """Confidence ellipse for 2D data."""
    if x.size != y.size:
        raise ValueError("x and y must have the same size")
    cov = np.cov(x, y)
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
    ell_radius_x = np.sqrt(1 + pearson)
    ell_radius_y = np.sqrt(1 - pearson)
    ellipse = Ellipse((0, 0), width=ell_radius_x * 2, height=ell_radius_y * 2,
                      facecolor=facecolor, **kwargs)
    scale_x = np.sqrt(cov[0, 0]) * n_std
    mean_x = np.mean(x)
    scale_y = np.sqrt(cov[1, 1]) * n_std
    mean_y = np.mean(y)
    transf = transforms.Affine2D().rotate_deg(45).scale(scale_x, scale_y).translate(mean_x, mean_y)
    ellipse.set_transform(transf + ax.transData)
    return ax.add_patch(ellipse)


def plot_precision_heatmaps(all_prec_results, vmin=0, vmax=300, cmap="Reds"):
    """
    3×3 heatmap of horizontal and vertical error per validation point,
    for each subject/session. Replicates precision_analysis.ipynb cell 18.
    """
    keys = list(all_prec_results.keys())
    n = len(keys)
    fig, axes = plt.subplots(n, 2, figsize=(5, 2.5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for row_idx, key in enumerate(keys):
        df_res = all_prec_results[key]
        unique_xs = sorted(df_res["presented_point_x"].unique())
        unique_ys = sorted(df_res["presented_point_y"].unique(), reverse=True)

        for col_idx, (error_col, label) in enumerate([
            ("horizontal_error_px", "Horizontal"),
            ("vertical_error_px", "Vertical"),
        ]):
            grid = np.full((len(unique_ys), len(unique_xs)), np.nan)
            for i, py in enumerate(unique_ys):
                for j, px in enumerate(unique_xs):
                    mask = (df_res["presented_point_x"] == px) & (df_res["presented_point_y"] == py)
                    val = df_res.loc[mask, error_col].mean()
                    grid[i, j] = val

            ax = axes[row_idx, col_idx]
            im = ax.imshow(grid, cmap=cmap, vmin=vmin, vmax=vmax, aspect="equal")
            ax.set_title(f"{key}\n{label} (px)", fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])
            plt.colorbar(im, ax=ax, shrink=0.8)

    plt.suptitle("Precision error per validation point", fontsize=10, fontweight="bold")
    plt.tight_layout()
    plt.show()


plot_precision_heatmaps(all_prec_results)


### 5 Error over the experiment

Error scatter per trial with regression, coloured by block.  
Replicates the `lmplot` from the original `precision_analysis.ipynb`.


In [ ]:
from scipy.stats import linregress


def plot_precision_over_time(all_prec_results):
    """
    Precision error over the course of the experiment per subject/session.
    Replicates precision_analysis.ipynb cell 19.
    """
    palette = sns.color_palette("flare", 8)

    for key, df_res in all_prec_results.items():
        df_plot = df_res.dropna(subset=["horizontal_error_px"]).copy().reset_index(drop=True)
        df_plot["abs"] = df_plot.index + 1
        df_plot["block"] = ((df_plot["trial_index"] - 1) % 8 + 1)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        for ax, (error_col, ylabel) in zip(axes, [
            ("horizontal_error_px", "Horizontal error (px)"),
            ("vertical_error_px", "Vertical error (px)"),
            ("total_error_px", "Total error (px)"),
        ]):
            for block_id in sorted(df_plot["block"].unique()):
                sub = df_plot[df_plot["block"] == block_id]
                color = palette[(int(block_id) - 1) % len(palette)]
                ax.scatter(sub["abs"], sub[error_col], c=[color], alpha=0.4, s=8)

            slope, intercept, *_ = linregress(df_plot["abs"], df_plot[error_col])
            x_range = np.array([df_plot["abs"].min(), df_plot["abs"].max()])
            ax.plot(x_range, slope * x_range + intercept, "k-", linewidth=2)

            ax.vlines([i * 72 for i in range(1, 9)], 0, 600, colors="k",
                      linewidths=0.5, linestyles="dashed")
            ax.set_xlim(0, len(df_plot) + 5)
            ax.set_ylim(0, 600)
            ax.set_xlabel("Trial")
            ax.set_ylabel(ylabel)
            ax.set_title(f"{key}")

        plt.suptitle(f"Precision error — {key}", fontsize=11, fontweight="bold")
        plt.tight_layout()
        plt.show()

        h = df_res['horizontal_error_px'].mean()
        v = df_res['vertical_error_px'].mean()
        t = df_res['total_error_px'].mean()
        print(f"{key}: H={h:.1f} px  V={v:.1f} px  Total={t:.1f} px")


plot_precision_over_time(all_prec_results)


### 6 Gaze scatter vs presented point

For each validation point, plots all gaze samples  
with confidence ellipses — same as the original notebook.


In [ ]:
def plot_gaze_scatter(all_prec_results, screen_res=(1920, 1080)):
    """
    Gaze scatter vs presented validation point, with confidence ellipses (2 std).
    Reads entirely from pyxations feather — no CSV needed.
    """
    for key, df_res in all_prec_results.items():
        subject_id = df_res['subject_id'].iloc[0]
        session    = df_res['session'].iloc[0]

        sub_path = prec_derivatives_path / f"sub-{subject_id}"
        samples  = feather.read_feather(sub_path / f"ses-{session}" / "samples.feather")

        val = samples[samples['trial-tag'] == 'validation-stimulus'].copy()

        unique_pts = (
            val[['start-x', 'start-y', 'center_x', 'center_y']]
            .drop_duplicates(['start-x', 'start-y'])
            .reset_index(drop=True)
        )
        unique_pts['px'] = unique_pts['start-x'] + unique_pts['center_x']
        unique_pts['py'] = unique_pts['start-y'] + unique_pts['center_y']

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.set_xlim(0, screen_res[0])
        ax.set_ylim(screen_res[1], 0)  # y=0 at top
        ax.axvline(screen_res[0] / 2, color="gray", linewidth=0.5, linestyle="--")
        ax.axhline(screen_res[1] / 2, color="gray", linewidth=0.5, linestyle="--")

        colors = plt.cm.tab10(np.linspace(0, 1, len(unique_pts)))
        for color, (_, pt) in zip(colors, unique_pts.iterrows()):
            pt_samples = val[
                (val['start-x'] == pt['start-x']) & (val['start-y'] == pt['start-y'])
            ]
            xs = pt_samples['X'].values
            ys = pt_samples['Y'].values

            ax.scatter(xs, ys, s=3, alpha=0.2, color=color)
            ax.scatter(pt['px'], pt['py'], marker='+', s=200, linewidths=2, color=color,
                       label=f"({pt['px']:.0f}, {pt['py']:.0f})")

            if len(xs) > 2:
                confidence_ellipse(xs, ys, ax, n_std=2, edgecolor=color, linewidth=1.5)

        ax.set_title(f"{key} — gaze vs presented point", fontsize=10)
        ax.set_xlabel("X (px)")
        ax.set_ylabel("Y (px)")
        ax.legend(fontsize=7, loc="upper right")
        plt.tight_layout()
        plt.show()


plot_gaze_scatter(all_prec_results)

---
## Notes

- `_staging/` is temporary and can be deleted after BIDS generation.
- `_derivatives/` contains all pyxations output (one feather per subject/session).
- To re-run detection: pass `overwrite=True` to `compute_derivatives_for_dataset`.
- `tSample` in the pyxations feather = time relative to trial onset (ms),
  equivalent to the `t` field in the WebGazer JSON. The `first_sample=500` filter
  is identical to the original `precision_analysis.ipynb`.
